In [ ]:
from pathlib import Path
import sys

STYLE_ROOT = Path.cwd()
if not (STYLE_ROOT / "scripts" / "plot_style.py").is_file():
    STYLE_ROOT = STYLE_ROOT.parent
if not (STYLE_ROOT / "scripts" / "plot_style.py").is_file():
    raise FileNotFoundError("Start the notebook in the repository root or scripts/.")
sys.path.insert(0, str(STYLE_ROOT))
sys.path.insert(0, str(STYLE_ROOT / "scripts"))
# Reload edits from disk when rerunning this cell in an existing kernel.
import importlib
from scripts import plot_style
importlib.reload(plot_style)
from scripts.plot_style import (
    apply_style, figure_size, export_figure, bottom_legend, FONT_SIZE, STROKES,
)
apply_style()

from matplotlib.gridspec import GridSpec
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

from plot_iqm import smooth_task_aware_ema


def plot_seed_comparisons(
    seeds: list[int],
    save: bool,
    methods: list[str] | str,
    heatmap_path_templates: list[str] | str,
    timesteps_per_task: list[int],
    vmax: list[int] | int,
    prefix: str = "",
    show_colorbars: bool = True,
    training_curve_path_templates: list[str] | str | None = None,
    smooth: int = 1,
):
    """
    Generates one figure containing all seed comparisons.

    Each seed has one heatmap row, with methods in columns. If training
    curve paths are supplied, a matching plot row appears below each heatmap
    row. When `show_colorbars` is True, every seed has a vertical colorbar.
    """
    if isinstance(vmax, int):
        vmax = [vmax] * (1 if isinstance(methods, str) else len(methods))

    if isinstance(methods, str):
        methods = [methods]

    if isinstance(heatmap_path_templates, str):
        heatmap_path_templates = [heatmap_path_templates]

    if len(methods) != len(heatmap_path_templates):
        raise ValueError(
            "`methods` and `heatmap_path_templates` must have the same length."
        )

    if len(vmax) != len(methods):
        raise ValueError("`vmax` must have one value per method.")

    plot_training_curves = training_curve_path_templates is not None
    if isinstance(training_curve_path_templates, str):
        training_curve_path_templates = [training_curve_path_templates]
    if plot_training_curves and len(training_curve_path_templates) != len(methods):
        raise ValueError("`training_curve_path_templates` must have one path per method.")

    if not seeds:
        raise ValueError("At least one seed must be provided.")

    num_rows = len(seeds)
    num_methods = len(methods)

    # Add one GridSpec column only when colorbars are enabled.
    num_grid_cols = num_methods + int(show_colorbars)


    fig = plt.figure(
        figsize=figure_size("heatmap_curve" if plot_training_curves else "heatmap", rows=num_rows, cols=num_methods),
    )

    width_ratios = [1] * num_methods
    if show_colorbars:
        width_ratios.append(0.08)

    rows_per_seed = 2 if plot_training_curves else 1
    gs = GridSpec(
        num_rows * rows_per_seed,
        num_grid_cols,
        figure=fig,
        width_ratios=width_ratios,
        height_ratios=([5, 1.7] if plot_training_curves else [5]) * num_rows,
        wspace=0.05,
        hspace=0.25 if plot_training_curves else 0.1,
    )

    heatmap_axes = [
        [fig.add_subplot(gs[row * rows_per_seed, col]) for col in range(num_methods)]
        for row in range(num_rows)
    ]
    curve_axes = (
        [[fig.add_subplot(gs[row * 2 + 1, col]) for col in range(num_methods)]
         for row in range(num_rows)]
        if plot_training_curves else []
    )

    if show_colorbars:
        colorbar_axes = [
            fig.add_subplot(gs[row * rows_per_seed, -1])
            for row in range(num_rows)
        ]
    else:
        colorbar_axes = []

    for row_idx, seed in enumerate(seeds):
        row_rightmost_im = None

        for col_idx, (method, heatmap_path_template, cur_vmax) in enumerate(
            zip(methods, heatmap_path_templates, vmax)
        ):
            ax_heat = heatmap_axes[row_idx][col_idx]

            method_label = (
                method.split("/")[-1]
                .replace("_", " ")
                .replace("-", " ")
            )
            method_label = method_label.replace("<s>", str(seed))

            heatmap_path = heatmap_path_template.format(seed=seed)
            is_empty = False

            try:
                df_heat = pd.read_csv(heatmap_path)
                df_heat.columns = ["alpha", "beta", "value"]

                pivot_df = df_heat.pivot(
                    index="beta",
                    columns="alpha",
                    values="value",
                )
            except Exception:
                pivot_df = pd.DataFrame(
                    np.nan,
                    index=np.linspace(-0.5, 1.5, 200),
                    columns=np.linspace(-0.5, 1.5, 200),
                )
                is_empty = True

            im = ax_heat.imshow(
                pivot_df,
                origin="lower",
                vmin=0,
                vmax=cur_vmax,
            )

            if col_idx == num_methods - 1:
                row_rightmost_im = im

            if is_empty:
                ax_heat.text(
                    0.5,
                    0.5,
                    "No Data Available",
                    ha="center",
                    va="center",
                    color="gray",
                    fontsize=FONT_SIZE,
                    transform=ax_heat.transAxes,
                )

            label_data = [
                (50, 150, r"$\theta_{1}$"),
                (150, 150, r"$\theta_{2}$"),
                (150, 50, r"$\theta_{3}$"),
            ]

            x_coords = [point[0] for point in label_data]
            y_coords = [point[1] for point in label_data]

            ax_heat.plot(
                x_coords,
                y_coords,
                color="red",
                linestyle="--",
                linewidth=STROKES["curve"],
            )

            ax_heat.set_xticks([])
            ax_heat.set_yticks([])

            for x, y, label in label_data:
                ax_heat.text(
                    x,
                    y,
                    label,
                    color="white",

                    ha="center",
                    va="center",
                    bbox={
                        "boxstyle": "round,pad=0.3",
                        "facecolor": "black",
                        "edgecolor": "none",
                        "alpha": 0.6,
                    },
                )

            if row_idx == 0:
                ax_heat.set_title(
                    method_label,
                    pad=10,
                    fontsize=FONT_SIZE,
                )

            if col_idx == 0:
                ax_heat.set_ylabel(
                    f"Seed {seed}",
                    fontsize=FONT_SIZE,
                    labelpad=10,
                )

            if plot_training_curves:
                ax_curve = curve_axes[row_idx][col_idx]
                curve_path = training_curve_path_templates[col_idx].format(seed=seed)
                reward_columns = [f"eval/random/{task}/mean_reward" for task in range(1)]
                time_columns = [f"time/{task}/total_timesteps" for task in range(1)]
                try:
                    df_curve = pd.read_csv(
                        curve_path,
                        usecols=lambda column: column in reward_columns
                        or column in time_columns,
                    )
                except (OSError, ValueError, pd.errors.EmptyDataError):
                    df_curve = pd.DataFrame()

                for task, (time_column, reward_column) in enumerate(
                    zip(time_columns, reward_columns), start=1
                ):
                    if time_column not in df_curve or reward_column not in df_curve:
                        continue
                    points = df_curve[[time_column, reward_column]].dropna()
                    if not points.empty:
                        timesteps = points[time_column] / 100_000
                        max_timesteps = max(timesteps)

                        data = smooth_task_aware_ema(points[reward_column], points[time_column], smooth, timesteps_per_env=timesteps_per_task[col_idx], num_tasks=3)

                        ax_curve.plot(
                            timesteps,
                            data / cur_vmax,
                            label=f"Task {task}",
                            color="#E69F00",
                            linewidth=STROKES["curve"],
                        )
                        ax_curve.set_yticks([0, 1], labels=["0", "1"])

                        task_boundaries = np.arange(0, max_timesteps + 1, timesteps_per_task[col_idx] / 100_000)
                        ax_curve.set_xticks(task_boundaries)

                        for boundary in task_boundaries[1:]:
                            if boundary < max_timesteps:
                                ax_curve.axvline(boundary, color="0.4", linewidth=STROKES["task_boundary"], linestyle="--", zorder=0)
                        ax_curve.set_xticks(task_boundaries)

                if ax_curve.lines:
                    pass

                else:
                    ax_curve.text(
                        0.5, 0.5, "No Training Data Available",
                        ha="center", va="center", color="gray",
                        transform=ax_curve.transAxes,
                    )
                ax_curve.set_xlabel("Training timesteps")
                if col_idx == 0:
                    ax_curve.set_ylabel("Mean reward", fontsize=FONT_SIZE)
                ax_curve.grid(alpha=0.25)

        # ======================================================
        # Manual colorbar for this row
        # ======================================================
        if show_colorbars and row_rightmost_im is not None:
            cax = colorbar_axes[row_idx]

            gradient = np.linspace(0, vmax[0], 512).reshape(-1, 1)

            cax.imshow(
                gradient,
                cmap=row_rightmost_im.get_cmap(),
                norm=row_rightmost_im.norm,
                aspect="auto",
                origin="lower",
            )

            cax.set_xticks([])

            cax.set_yticks(
                np.linspace(0, 511, 5),
                labels=[0, 25, 50, 75, 100],
            )

            cax.yaxis.tick_right()
            cax.tick_params(
                axis="y",
                labelsize=FONT_SIZE,
                length=0,
                pad=8,
            )

            for spine in cax.spines.values():
                spine.set_visible(False)

    if save:
        export_figure(fig, 
            f"{STYLE_ROOT}/output/plots/heatmaps/{prefix}_{'heatmaps_and_curves' if plot_training_curves else 'heatmaps'}.pdf",
            format="pdf",
        )

    plt.show()

In [ ]:
if __name__ == "__main__":
    plot_seed_comparisons(
        seeds=[11],
        methods=[

            "DDPG",


        ],
        heatmap_path_templates=[

            str(STYLE_ROOT / "output/interpolations/joint_incremental/ji-ddpg/data_{seed}.csv"),


        ],
        # Set to None to show only the heatmaps.
        training_curve_path_templates=[

            str(STYLE_ROOT / "output/joint_incremental/ji-ddpg/ji-ddpg-s_{seed}.csv"),


        ],
        save=True,
        prefix='ddpg-reset-smooth-seed-11',

        timesteps_per_task=[150_000],

        vmax=[1000],

        show_colorbars=False,
        smooth=30,

    )

In [ ]:
from plot_iqm import smooth_task_aware_ema
from common import interquartile_mean

def plot_avg_heatmap(
    seeds: list[int],
    save: bool,
    methods: list[str] | str,
    heatmap_path_templates: list[str] | str,
    vmax: list[int] | int,
    prefix: str = "",
    show_colorbars: bool = True,
    use_iqm: bool = False,
):
    """
    Plot the mean or interquartile mean (IQM) heatmap across seeds for each method.

    Set `use_iqm=True` to average the middle 50% of seed values at each
    grid point. Rewards are divided by each method’s `vmax`, so all heatmaps
    share a 0–1 color scale and one optional colorbar on the right.
    """
    if isinstance(vmax, int):
        vmax = [vmax] * (1 if isinstance(methods, str) else len(methods))

    if isinstance(methods, str):
        methods = [methods]

    if isinstance(heatmap_path_templates, str):
        heatmap_path_templates = [heatmap_path_templates]

    if len(methods) != len(heatmap_path_templates):
        raise ValueError(
            "`methods` and `heatmap_path_templates` must have the same length."
        )

    if len(vmax) != len(methods):
        raise ValueError("`vmax` must have one value per method.")

    if not seeds:
        raise ValueError("At least one seed must be provided.")

    num_methods = len(methods)
    if not num_methods:
        raise ValueError("At least one method must be provided.")

    fig = plt.figure(figsize=figure_size("heatmap_row", cols=num_methods))
    width_ratios = [1] * num_methods + ([0.08] if show_colorbars else [])
    gs = GridSpec(
        1,
        len(width_ratios),
        figure=fig,
        width_ratios=width_ratios,
        wspace=0.05,
        hspace=0.1,
        left=0.035, right=0.96, bottom=0.08, top=0.78,
    )
    heatmap_axes = [fig.add_subplot(gs[0, index]) for index in range(num_methods)]
    colorbar_ax = fig.add_subplot(gs[0, -1]) if show_colorbars else None

    for plot_idx, (method, heatmap_path_template, cur_vmax) in enumerate(
            zip(methods, heatmap_path_templates, vmax)
        ):
        data = []
        for seed in seeds:
            heatmap_path = heatmap_path_template.format(seed=seed)
            df_heat = pd.read_csv(heatmap_path)
            df_heat.columns = ["alpha", "beta", "value"]
            df_heat.insert(0, 'seed', seed)
            data.append(df_heat)
        df = pd.concat(data, ignore_index=True)

        if use_iqm:
            df = df.groupby(['alpha', 'beta'], as_index=False)['value'].agg(
                lambda values: interquartile_mean(values.dropna().to_numpy())
                if values.notna().any() else np.nan
            )
        else:
            df = df.groupby(['alpha', 'beta'], as_index=False)['value'].mean()

        pivot_df = df.pivot(
            index="beta",
            columns="alpha",
            values="value",
        )

        ax_heat = heatmap_axes[plot_idx]
        im = ax_heat.imshow(
            pivot_df / cur_vmax,
            origin="lower",
            vmin=0,
            vmax=1,
        )

        label_data = [
            (50, 150, r"$\theta_{1}$"),
            (150, 150, r"$\theta_{2}$"),
            (150, 50, r"$\theta_{3}$"),
        ]

        x_coords = [point[0] for point in label_data]
        y_coords = [point[1] for point in label_data]

        ax_heat.plot(
            x_coords,
            y_coords,
            color="red",
            linestyle="--",
            linewidth=STROKES["curve"],
        )

        ax_heat.set_xticks([])
        ax_heat.set_yticks([])

        ax_heat.scatter(x_coords, y_coords, s=24, color="red", zorder=4)

        label_offsets = [(-3, 3), (3, 3), (3, -3)]
        for (x, y, label), (dx, dy) in zip(label_data, label_offsets):
            ax_heat.annotate(
                label,
                xy=(x, y),
                xytext=(dx, dy),
                textcoords="offset points",
                color="white",
                fontsize=FONT_SIZE,
                ha="right" if dx < 0 else "left",
                va="bottom" if dy > 0 else "top",
                zorder=5,
                bbox={
                    "boxstyle": "round,pad=0.3",
                    "facecolor": "black",
                    "edgecolor": "none",
                    "alpha": 0.6,
                },
            )


        ax_heat.set_title(
            method,
            pad=10,
            fontsize=FONT_SIZE,
        )
        ax_heat.annotate(
            f"({chr(ord('a') + plot_idx)})",
            xy=(0, 1), xytext=(0, 6),
            xycoords="axes fraction", textcoords="offset points",
            ha="left", va="bottom", fontweight="bold",
            fontsize=FONT_SIZE,
        )

    if show_colorbars:
        colorbar = fig.colorbar(im, cax=colorbar_ax, ticks=[0, 1])
        colorbar.ax.set_yticklabels(["0", "1"])
        colorbar.ax.tick_params(labelsize=FONT_SIZE)

    if save:
        export_figure(fig, 
            f"{STYLE_ROOT}/output/plots/heatmaps/{prefix}_heatmap_{'iqm' if use_iqm else 'mean'}.pdf",
            format="pdf",
        )

    plt.show()

In [ ]:
if __name__ == "__main__":
    plot_avg_heatmap(
        seeds=list(range(50)),
        methods=[
            "DQN",
            "SAC-D",
            "DDPG",
            "SAC",
        ],
        heatmap_path_templates=[
            str(STYLE_ROOT / "output/interpolations/joint_incremental/ji-dqn/data_{seed}.csv"),
            str(STYLE_ROOT / "output/interpolations/joint_incremental/ji-sacd-l_00001/data_{seed}.csv"),
            str(STYLE_ROOT / "output/interpolations/joint_incremental/ji-ddpg/data_{seed}.csv"),
            str(STYLE_ROOT / "output/interpolations/joint_incremental/ji-sac/data_{seed}.csv"),
        ],
        save=True,
        prefix='reset-iqm',
        vmax=[500, 500, 1000, 1000, ],
        show_colorbars=True,
        use_iqm=True,
    )